# CCA-F Review Notebook — Context Management

This notebook focuses on Domain 5: context management. Each task keeps the required structure: concept explanation, runnable Python example, exam summary, and two official-style practice questions. All examples are local simulations and require no API key.


## Environment Setup

In [ ]:
import json
import re
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any, Optional

MODEL = "claude-haiku-4-5-20251001"
print("Environment ready: examples are local simulations shaped like Claude API and Claude Code workflows.")

## D5.1 — Manage long-context inputs and the 'lost in the middle' effect

### 📖 Concept Explanation

Long context does not mean uniformly reliable attention. Models often show **primacy and recency bias**: content near the beginning and end is used more reliably than evidence buried in the middle. This is the practical risk behind the lost-in-the-middle effect.

For CCA-F, the engineering answer is not simply “use a larger context window.” Put the task objective, key facts, and decision criteria at the top. Use clear section headings. Restate output constraints and hard requirements near the end. Trim verbose tool results before adding them to the prompt, keeping only fields needed for the current decision. Raw logs, full HTML, unfiltered search results, and repeated chat history increase cost and can dilute the evidence that matters.


In [ ]:
# Task 5.1: long context, lost in the middle, key placement, and tool-output trimming.
# This scoring function simulates the risk: evidence near the edges is easier to recover than middle evidence.

verbose_tool_result = {
    **{f"debug_field_{i}": f"noise_{i}" for i in range(18)},
    "order_id": "ORD-9",
    "refund_status": "eligible",
    "policy_basis": "returned_within_30_days",
    "amount_usd": 88.10,
}

def trim_refund_tool_result(raw: dict) -> dict:
    """Keep only the fields needed for the automatic-refund decision."""
    keep = ["order_id", "refund_status", "policy_basis", "amount_usd"]
    return {key: raw[key] for key in keep}

def position_score(sections: list[tuple[str, str]], keyword: str) -> float:
    """Demonstrate lost-in-the-middle risk with lower weight for middle sections."""
    n = len(sections)
    score = 0.0
    for index, (_, text) in enumerate(sections):
        if keyword in text:
            distance_to_edge = min(index, n - 1 - index)
            score += 1 / (1 + distance_to_edge)
    return round(score, 3)

bad_context = [
    ("chat_history", "The customer asks about a refund."),
    ("raw_tool_output", json.dumps(verbose_tool_result)),
    ("irrelevant_logs", "many debug logs..." * 20),
    ("critical_fact", "ORD-9 refund_status=eligible policy_basis=returned_within_30_days"),
    ("closing", "Please answer."),
]

good_context = [
    ("decision_summary", "Task: decide whether ORD-9 can be automatically refunded. Key fact: refund_status=eligible."),
    ("trimmed_tool_result", json.dumps(trim_refund_tool_result(verbose_tool_result))),
    ("background", "Only concise refund-relevant background is retained."),
    ("final_constraints", "Use only order_id, refund_status, policy_basis, and amount_usd."),
]

print("bad_context_score:", position_score(bad_context, "refund_status=eligible"))
print("good_context_score:", position_score(good_context, "refund_status=eligible"))
print(json.dumps(dict(good_context), indent=2))


### ⚠️ Exam Summary & Common Traps

**✅ Correct patterns**

- Put the objective, key facts, and decision criteria near the top of the context.
- Restate output format, constraints, and prohibitions near the end to benefit from recency.
- Trim tool output at the field level instead of pasting raw logs, HTML, or broad search results.
- Use headings and structured JSON to separate facts, background, constraints, and open tasks.

**✗ Common traps**

- Assuming a larger context window eliminates lost-in-the-middle behavior.
- Burying the only critical evidence in the middle of a long prompt.
- Keeping every tool field “for completeness,” which raises cost and noise.
- Trimming chat history but forgetting to trim verbose tool responses.


### 🧪 Practice Questions

**Q1.** An agent must decide whether to issue an automatic refund. A tool returns 80 fields, but only `order_id`, `refund_status`, `policy_basis`, and `amount_usd` matter. What is the best context strategy?

A) Put a key summary at the top, trim the tool result, and restate the decision fields near the end  
B) Keep the complete tool result so no information is lost  
C) Place the key fields in the middle to preserve original ordering  
D) Only tell the model, “Pay attention to important information”

> **Answer: A**  
> **Explanation:** CCA-F emphasizes layout and trimming for reliability. Full tool output can dilute key evidence; critical facts should be front-loaded and reinforced by final constraints.

**Q2.** Which statement about long context and lost in the middle is correct?

A) Once the context window is large enough, middle evidence is as reliable as beginning evidence  
B) Critical evidence should be placed near the beginning or end, with headings, summaries, and structured fields  
C) More complete tool output always improves business correctness  
D) Lost in the middle can only be solved by switching to a larger model

> **Answer: B**  
> **Explanation:** Long context still has positional attention risks. Engineers reduce the risk with placement, summarization, trimming, and structured context.


## D5.2 — Design escalation patterns and confidence calibration

### 📖 Concept Explanation

Escalation and confidence calibration are control-plane mechanisms for reliable agents. High-risk workflows cannot rely on a model saying it is confident. Escalation triggers should be explicit rules: the user asks for a human, the policy is missing or ambiguous, permissions are insufficient, identity or compliance risk exists, repeated attempts make no progress, multiple candidate records cannot be disambiguated, or a required tool is unavailable with no safe recovery path.

CCA-F often tests the distinction between **sentiment and complexity**. An angry user is not automatically a complex case, and a calm user can still present a high-risk ambiguity. Sentiment can influence priority and tone, but it should not be the sole escalation signal. Confidence thresholds should be calibrated on labeled data: when the model reports high confidence, how often is it actually correct? If the threshold is not met, use graceful degradation: state the limitation, preserve known facts, collect missing information, or hand off to a human with structure.


In [ ]:
# Task 5.2: escalation, confidence calibration, HitL thresholds, and graceful degradation.
# Sentiment is not complexity, and self-reported confidence is not calibrated accuracy.

calibration_rows = [
    {"case": "simple_refund", "self_confidence": 0.93, "correct": True},
    {"case": "policy_exception", "self_confidence": 0.91, "correct": False},
    {"case": "address_change", "self_confidence": 0.82, "correct": True},
    {"case": "ambiguous_identity", "self_confidence": 0.88, "correct": False},
]

def calibrated_precision_at(threshold: float) -> float | None:
    selected = [row for row in calibration_rows if row["self_confidence"] >= threshold]
    if not selected:
        return None
    return sum(row["correct"] for row in selected) / len(selected)

def decide_escalation(
    *,
    user_text: str,
    sentiment: str,
    task_complexity: str,
    policy_gap: bool,
    candidate_matches: int,
    attempts_without_progress: int,
    model_confidence: float,
) -> dict:
    reasons = []
    if re.search(r"human|agent|representative", user_text, re.I):
        reasons.append("explicit_human_request")
    if policy_gap:
        reasons.append("policy_gap")
    if candidate_matches > 1:
        reasons.append("ambiguous_multiple_matches")
    if attempts_without_progress >= 2:
        reasons.append("no_progress_after_retries")
    if task_complexity == "high" and model_confidence < 0.90:
        reasons.append("below_hitl_threshold_for_high_complexity")

    # Sentiment affects service priority, but it does not independently decide escalation.
    priority = "urgent" if sentiment == "angry" and reasons else "normal"
    action = "human_handoff" if reasons else "autonomous_resolution"
    fallback = "State limits, preserve facts, collect missing fields, or hand off." if reasons else "Continue automated handling."
    return {"action": action, "priority": priority, "reasons": reasons, "fallback": fallback}

print("precision_at_0.90:", calibrated_precision_at(0.90))
print(json.dumps(decide_escalation(
    user_text="I am calm, but this email matches two customer accounts",
    sentiment="calm",
    task_complexity="high",
    policy_gap=False,
    candidate_matches=2,
    attempts_without_progress=0,
    model_confidence=0.87,
), indent=2))


### ⚠️ Exam Summary & Common Traps

**✅ Correct patterns**

- Escalate explicit human requests, policy gaps, insufficient permissions, repeated no-progress loops, and unresolved ambiguity.
- Set HitL thresholds from task risk and labeled calibration data, not model self-description.
- Use sentiment for priority and tone, not as a replacement for complexity or risk assessment.
- Graceful degradation should state limits, preserve state, propose next steps, or create a structured handoff.

**✗ Common traps**

- Escalating because the user is angry, or not escalating because the user is calm.
- Treating `confidence: 0.95` as an actual calibrated probability.
- Continuing automated troubleshooting after the user explicitly requests a human.
- Saying “try again later” after tool failure without preserving context or recovery options.


### 🧪 Practice Questions

**Q1.** A user is calm, but one email address matches two customer accounts and the automated flow cannot determine which account is correct. What should the agent do?

A) Escalate or trigger HitL because the case contains unresolved high-risk ambiguity  
B) Avoid escalation because the user is calm  
C) Randomly choose the account with the most recent login  
D) Ask the model to choose whichever account feels more likely

> **Answer: A**  
> **Explanation:** Complexity and risk come from failed identity disambiguation, not sentiment. Multiple unresolved matches require escalation or human confirmation.

**Q2.** Which statement about model confidence best matches CCA-F guidance?

A) If the model reports high confidence, high-risk actions can be automated  
B) Confidence thresholds should be calibrated with labeled data and adjusted for task risk and HitL requirements  
C) Sentiment scores are better than calibration data for escalation decisions  
D) A detailed model explanation removes the need for a human escalation path

> **Answer: B**  
> **Explanation:** Self-reported confidence is not a reliable probability. Production systems need calibration data and risk-sensitive human intervention thresholds.


## D5.3 — Implement error propagation and information provenance across agents

### 📖 Concept Explanation

Reliable multi-agent systems propagate errors and provenance across boundaries. The goal is not for every subagent to succeed; the goal is for the coordinator to understand what succeeded, what failed, what was attempted, what partial evidence exists, and which claims are supported by which sources.

A failed subagent should return `failure_type`, `attempted`, `partial_results`, `retryable`, and `alternatives`. A successful subagent should attach provenance to each claim: source, excerpt, date or page number, and attribution to the producing agent. The coordinator should mark coverage gaps instead of silently converting missing evidence into certainty. Complete handoff includes the current goal, attempted actions, failure reasons, partial evidence, next actions, and risks.


In [ ]:
# Task 5.3: error propagation, provenance, attribution, and handoff completeness.
# Failure must not be swallowed; successful claims must remain traceable to sources.

subagent_outputs = [
    {
        "agent": "web_search",
        "status": "failed",
        "failure_type": "timeout",
        "attempted": ["query: AI music market 2026", "query: enterprise adoption"],
        "partial_results": [{"title": "Market briefing", "url": "https://example.test/briefing"}],
        "retryable": True,
        "alternatives": ["retry with a narrower query", "ask the analyst to upload the source report"],
    },
    {
        "agent": "doc_analysis",
        "status": "success",
        "claims": [
            {
                "claim": "The policy allows refunds within 30 days.",
                "source": "refund_policy.pdf#page=4",
                "excerpt": "Refunds are permitted within 30 days of delivery.",
                "attribution": "doc_analysis",
            }
        ],
    },
]

def build_handoff(outputs: list[dict]) -> dict:
    claims = []
    coverage_gaps = []
    next_actions = []

    for item in outputs:
        if item["status"] == "success":
            for claim in item.get("claims", []):
                required = {"claim", "source", "attribution"}
                missing = sorted(required - set(claim))
                claims.append({**claim, "provenance_complete": not missing, "missing": missing})
        else:
            coverage_gaps.append({
                "agent": item["agent"],
                "failure_type": item["failure_type"],
                "attempted": item.get("attempted", []),
                "partial_results": item.get("partial_results", []),
                "retryable": item.get("retryable", False),
            })
            next_actions.extend(item.get("alternatives", []))

    return {"claims": claims, "coverage_gaps": coverage_gaps, "recommended_next_actions": next_actions}

print(json.dumps(build_handoff(subagent_outputs), indent=2))


### ⚠️ Exam Summary & Common Traps

**✅ Correct patterns**

- Failed subagents return structured failure type, attempted actions, partial results, retryability, and alternatives.
- Each claim keeps source, excerpt or page, producing agent, and other provenance needed for audit.
- The coordinator marks coverage gaps instead of presenting missing evidence as certainty.
- Handoffs include goal, status, evidence, failure reasons, next steps, and risk.

**✗ Common traps**

- Returning an empty success array when a subagent actually failed.
- Dropping sources during synthesis, making claims impossible to audit.
- Throwing away partial results because one subtask failed.
- Mixing source-backed facts and model inferences without attribution.


### 🧪 Practice Questions

**Q1.** A search subagent times out but found one potentially relevant report title before failing. What should it return?

A) `failure_type`, `attempted`, `partial_results`, `retryable`, and `alternatives`  
B) An empty success result so the coordinator can continue cleanly  
C) Only “search failed,” with no details  
D) A signal to terminate the entire multi-agent workflow

> **Answer: A**  
> **Explanation:** Structured failure context and partial results let the coordinator retry, choose an alternative path, continue with caveats, or escalate.

**Q2.** What best supports provenance and attribution in multi-agent synthesis?

A) Every claim carries its source, excerpt or page, and the agent that produced it  
B) The final summary keeps only conclusions and deletes all sources to save tokens  
C) Only the final coordinator agent is named  
D) A fluent final answer removes the need for a source chain

> **Answer: A**  
> **Explanation:** Provenance makes claims auditable and helps downstream systems distinguish facts, inferences, and evidence gaps.


## D5.4 — Optimize cost and latency with Batch API and prompt caching

### 📖 Concept Explanation

Batch API and prompt caching optimize different parts of the cost-latency tradeoff. Batch is for high-throughput, non-blocking, latency-tolerant offline work such as nightly extraction, evaluation runs, or historical backfills. It is not suitable for a user waiting in an interactive flow because results are not returned immediately.

Prompt caching is useful when many requests share a long, stable prefix: policies, schemas, tool instructions, or large reusable background. Caching volatile user-specific content has little benefit and can add complexity. CCA-F commonly tests the tradeoff: Batch can reduce unit cost but increases completion latency; prompt caching can reduce repeated-prefix cost and latency, but only when the prefix is long, stable, and reused. Real-time blocking requests favor synchronous APIs; offline throughput favors Batch.


In [ ]:
# Task 5.4: Batch API, prompt caching, blocking vs non-blocking flows, and cost tradeoffs.
# The multipliers below are illustrative for relative reasoning; production code should use current provider pricing.

from math import ceil

def choose_execution_plan(*, blocking: bool, requests: int, repeated_prefix_tokens: int, deadline_hours: float) -> dict:
    batch_candidate = (not blocking) and requests >= 1000 and deadline_hours >= 1
    cache_candidate = repeated_prefix_tokens >= 1024
    if batch_candidate:
        mode = "message_batches_api"
        latency_profile = "non_blocking_completion_window"
    else:
        mode = "synchronous_messages_api"
        latency_profile = "interactive_or_low_volume"
    return {"mode": mode, "use_prompt_cache": cache_candidate, "latency_profile": latency_profile}

def estimate_relative_cost(*, requests: int, input_tokens: int, repeated_prefix_tokens: int, use_batch: bool, use_cache: bool) -> dict:
    # Teaching-only relative cost: Batch discounts total work; caching discounts repeated prefix tokens.
    batch_multiplier = 0.50 if use_batch else 1.00
    cached_prefix_multiplier = 0.10 if use_cache else 1.00
    variable_tokens = max(input_tokens - repeated_prefix_tokens, 0)
    effective_input_tokens = repeated_prefix_tokens * cached_prefix_multiplier + variable_tokens
    relative_cost_units = requests * effective_input_tokens * batch_multiplier
    return {
        "effective_input_tokens_per_request": ceil(effective_input_tokens),
        "relative_cost_units": ceil(relative_cost_units),
        "batch_multiplier": batch_multiplier,
        "cached_prefix_multiplier": cached_prefix_multiplier,
    }

plan = choose_execution_plan(blocking=False, requests=50_000, repeated_prefix_tokens=8_000, deadline_hours=12)
cost = estimate_relative_cost(requests=50_000, input_tokens=10_000, repeated_prefix_tokens=8_000, use_batch=True, use_cache=True)

message_shape = {
    "model": MODEL,
    "custom_id": "doc-0001",
    "messages": [{"role": "user", "content": "Extract invoice fields from document 0001."}],
    "system": [{"type": "text", "text": "Long stable extraction schema and policy...", "cache_control": {"type": "ephemeral"}}],
}

print(json.dumps({"plan": plan, "cost": cost, "message_shape": message_shape}, indent=2))


### ⚠️ Exam Summary & Common Traps

**✅ Correct patterns**

- Use Batch for high-throughput, non-blocking, latency-tolerant offline work, with `custom_id` for result tracking.
- Use synchronous Messages API or streaming for blocking user-facing flows.
- Use prompt caching for long, stable, repeated prefixes such as policies, schemas, and tool instructions.
- Evaluate cost, latency, retries, monitoring, and failure recovery together, not just per-request price.

**✗ Common traps**

- Moving real-time chat, payment confirmation, or support handoff flows to Batch just to reduce cost.
- Caching volatile user input or short prompts with little reuse.
- Ignoring Batch’s non-immediate completion behavior and result-correlation requirements.
- Assuming prompt caching improves every request; it mainly helps repeated long prefixes.


### 🧪 Practice Questions

**Q1.** A system processes 50,000 contracts every night for next-morning delivery, and every request shares the same long extraction schema. What is the best fit?

A) Use Message Batches API and apply prompt caching to the stable schema  
B) Make the user interface synchronously wait for all contracts to finish  
C) Disable caching to avoid reusing any prefix  
D) Turn each contract into a multi-turn conversation to improve reliability

> **Answer: A**  
> **Explanation:** The workload is high-volume, non-blocking, and latency-tolerant, which fits Batch. The shared long stable schema fits prompt caching.

**Q2.** Which scenario is least appropriate for Batch API?

A) A user is waiting on a checkout page for payment-risk approval  
B) Generating 100,000 offline summaries overnight  
C) Backfilling historical document labels over a weekend  
D) Running a large evaluation set offline

> **Answer: A**  
> **Explanation:** Checkout risk approval is a blocking real-time flow. Batch is better for offline workloads that can tolerate delayed completion.
